# P3 프로젝트: 야생동물 자동 관측·분석 시스템

**핵심 질문:** *객체 탐지 성능(mAP)이 높은 모델은 야생동물 개체 수도 더 정확하게 셀까?*

이 프로젝트는 P3 Lab의 **CNN → 사전훈련 모델/전이학습(Xception) → 객체 탐지(Faster R-CNN)**를 실제 응용으로 발전시킨다. African Wildlife 사진에서 동물 4종(buffalo, elephant, rhino, zebra)을 탐지하고, **사진별 개체 수·종류별 출현 빈도·탐지 오류**를 분석한다.

**이 노트북의 원칙:** 모든 코드가 미리 완성되어 있다. `TODO`, `NotImplementedError`, 정답을 채워야 실행되는 셀은 **없다**. **학생의 일은 실행한 코드를 이해하고, 설정을 바꾸고, 실험을 직접 설계하며, 분석을 확장·검증하는 것**이다.

> **해석 범위:** 서로 다른 사진에 동일한 동물이 나타날 수 있으므로 탐지 수의 합은 **실제 야생동물 개체군 수가 아니다**. 시간·위치 정보 없이 이동 경로나 시간대별 출현 패턴도 추정하지 않는다.

**선택 심화 확장:** 본 실험을 마친 학생은 10장의 **동영상 객체 추적** 단원에서 학습한 YOLO로 연속 영상의 개체 ID를 추적하고 종류별 관측 횟수와 추적 ID 수를 비교할 수 있다. 동영상이 없어도 1~9장의 코드는 독립적으로 실행된다.

**코드 실행**

[(구글 코랩) P3 프로젝트: 야생동물 자동 관측·분석 시스템](https://colab.research.google.com/github/codingalzi/code-workout-dlp/blob/master/P3_project_wildlife.ipynb)에서 코드를 실행할 수 있다.

## 학습 목표와 제출물

1. **데이터 이해:** YOLO 정답 라벨과 탐지 클래스의 의미를 설명한다.
2. **전이학습:** 같은 사전훈련 YOLO에서 `freeze=10`과 `freeze=0`을 비교한다.
3. **모델 평가:** 두 모델의 **검증 mAP50-95와 Count MAE**를 함께 비교한다.
4. **응용 시스템:** 테스트 이미지에서 **사진별 개체 수, 객체별 탐지 기록, 동물 종류별 출현 사진 비율**을 생성한다.
5. **오류 분석:** 잘못 탐지된 사례를 시각적으로 검토하고 자동 관측 결과의 한계를 설명한다.
6. **동영상 심화(선택):** 학습된 YOLO에 객체 추적기를 결합하여 추적 ID, 영상 속 동물 관측 횟수, 이동 궤적을 분석한다.

**제출:** 실행된 `.ipynb` + 실험 비교표 + CSV/그래프 + 대표 오류 이미지 3장 + 결론 및 AI 활용 기록.

**진행 방식:** 먼저 **빠른 실행 모드**로 전체 흐름을 확인하고, 설정을 바꿔 **본 실험**을 수행한다. ‘빠른 실행’ 수치는 학습/평가 절차 연습용이며 최종 연구 결과로 사용하지 않는다.

> **본 실험의 핵심:** 처음에는 25 epoch로 학습하되, **중요한 것은 에포크 수 자체보다 검증 mAP50-95가 안정화되었는지**이다. 그래프에서 성능이 계속 향상된다면 학습 에포크를 늘려 다시 실험하고, 그 판단 근거를 기록한다.

### AI 활용과 자기주도학습

AI를 이용해 새 코드, 그래프, 분석을 만들 수 있다. 다만 **실제로 실행한 결과와 AI가 예상한 결과를 구분**하고, 제안받은 내용을 직접 확인한다.

프로젝트 중 **AI 활용 사례 3개**를 기록한다: *(1) 해결하려던 문제 (2) AI에 요청한 내용 (3) 채택·수정한 코드/설정 (4) 직접 확인한 결과와 판단*.

**권장 질문 예시:** “`freeze=10`이 어떤 층을 고정하는지 모델 구조와 연결해 설명해줘. 직접 확인할 수 있는 코드를 제안해줘.”

---
## 1. 환경 설정

아래 설치 셀은 Jupyter/Colab에서 실행한다. 처음 실행할 때 **라이브러리와 공개 가중치·데이터셋 다운로드에 인터넷**이 필요하다. GPU 사용을 권장하지만 CPU에서도 실행할 수 있다(시간이 오래 걸릴 수 있음).

> 기존 P3 Lab과 모델 구조·실험 비교를 맞추기 위해 **YOLO11n**을 사용한다. YOLO 버전 변경은 본 실험을 끝낸 뒤 선택적으로 해본다.

In [ ]:
%pip install -q ultralytics pandas matplotlib pillow

In [ ]:
from pathlib import Path
from collections import Counter
import json

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

# 프로젝트 설정: 학생이 가장 먼저 확인하고 변경할 부분
QUICK_MODE = True       # 전체 실행 확인 후 False로 바꿔 본 실험 수행
MODEL_NAME = "yolo11n.pt"
DATA_YAML = "african-wildlife.yaml"
SEED = 42

EPOCHS = 2 if QUICK_MODE else 25  # 본 실험: 먼저 25, 검증 mAP50-95가 계속 개선되면 50 또는 75 등으로 늘려 재실험
FRACTION = 0.20 if QUICK_MODE else 1.0
IMGSZ = 320 if QUICK_MODE else 512
BATCH = 8 if torch.cuda.is_available() else 4
DEVICE = 0 if torch.cuda.is_available() else "cpu"
THRESHOLD = 0.35

PROJECT = Path.cwd() / "P3_wildlife_project"
RUNS = PROJECT / ("quick_runs" if QUICK_MODE else "full_runs")
OUT = PROJECT / ("quick_outputs" if QUICK_MODE else "outputs")
RUNS.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

print("Mode:", "QUICK (practice)" if QUICK_MODE else "FULL (report)")
print("Ultralytics:", ultralytics.__version__)
print("Device:", DEVICE, "| Epochs:", EPOCHS, "| Train fraction:", FRACTION)

### 실험 1 — 실행 환경 점검

**학생 과제:** `QUICK_MODE=True`에서 끝까지 실행한 뒤, GPU/CPU, 에포크, 이미지 크기, 사용한 학습 이미지 비율을 기록하라. 본 실험은 `QUICK_MODE=False`로 설정해 같은 학습/검증/테스트 분할을 사용한다. **모든 모델에 동일한 설정**을 적용한다.

---
## 2. 데이터 탐색

African Wildlife 데이터의 네 클래스와 정답 라벨을 확인한다. YOLO 정답 파일의 한 줄은 `class_id x_center y_center width height` 형식이며, 좌표는 이미지 크기로 나눈 **정규화 좌표**다. 여기서는 어려운 좌표 변환 함수를 직접 구현하지 않고 이미지, 클래스, 개체 수의 관계를 먼저 확인한다.

In [ ]:
# 데이터 다운로드/경로 확인: Ultralytics 공식 데이터 정의를 사용
info = check_det_dataset(DATA_YAML)
ROOT = Path(info["path"])
names = info["names"]
CLASS_NAMES = [names[i] for i in range(len(names))] if isinstance(names, dict) else list(names)

# YOLO 데이터셋의 원래 train/val/test 분할 유지
images = {}
for split in ["train", "val", "test"]:
    folder = Path(info[split])
    images[split] = sorted(p for p in folder.rglob("*")
                           if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"])
    print(split, ":", len(images[split]), "images")
print("Classes:", CLASS_NAMES)


def read_ids(image_path):
    """사진 한 장에 있는 정답 객체의 클래스 번호를 반환한다."""
    label_file = ROOT / "labels" / image_path.parent.name / (image_path.stem + ".txt")
    if not label_file.exists():
        return []
    return [int(line.split()[0]) for line in label_file.read_text().splitlines() if line.strip()]

# 실제 이미지 4장을 확인
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, path in zip(axes, images["train"][:4]):
    ax.imshow(Image.open(path))
    ax.set_title(f"GT objects: {len(read_ids(path))}")
    ax.axis("off")
plt.tight_layout()

In [ ]:
# 클래스별 정답 객체 수 vs 그 동물이 등장한 사진 수
box_counts = Counter()
image_counts = Counter()
for path in images["train"]:
    ids = read_ids(path)
    box_counts.update(ids)          # 같은 사진에서 같은 동물이 여러 마리면 모두 셈
    image_counts.update(set(ids))   # 한 사진에는 해당 클래스가 등장했다고 1번만 셈

data_summary = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Objects": [box_counts[i] for i in range(len(CLASS_NAMES))],
    "Images": [image_counts[i] for i in range(len(CLASS_NAMES))],
})
display(data_summary)
data_summary.plot.bar(x="Class", y=["Objects", "Images"], rot=0, figsize=(8, 4),
                      title="Training annotations: objects vs images")
plt.tight_layout()

**학생 과제 2 — 데이터 이해 및 확장**

- 한 장에 얼룩말 3마리가 있으면 `Objects`와 `Images`가 각각 얼마씩 증가하는지 예를 들어 설명하라.
- **추가 분석:** 검증셋 또는 테스트셋에 대해서도 위 표를 직접 만들어 비교하라. 새 셀을 복사·수정해도 된다.
- 선택: 정답 라벨의 좌표를 이용해 원본 사진에 경계 상자를 그려보고 P3 Lab의 Faster R-CNN `boxes`와 비교하라.

---
## 3. 사전훈련 객체 탐지 모델 체험

P3 Lab에서는 Faster R-CNN의 `boxes`, `labels`, `scores`를 사용했다. Ultralytics YOLO에서는 `result.boxes.xyxy`, `result.boxes.cls`, `result.boxes.conf`에 해당한다.

**주의:** COCO로 학습된 기본 모델의 클래스와 African Wildlife의 네 클래스는 **같지 않다**. 기본 모델의 예측은 **정성적 관찰용**이며, 아래 네 클래스 mAP 비교에는 사용하지 않는다.

In [ ]:
base_model = YOLO(MODEL_NAME)
result = base_model.predict(str(images["train"][0]), conf=0.25,
                            imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
print("Detected objects:", len(result.boxes))
for box in result.boxes[:5]:
    class_id = int(box.cls.item())
    print(result.names[class_id], "confidence:", round(float(box.conf.item()), 3))
plt.figure(figsize=(7, 5))
plt.imshow(result.plot()[:, :, ::-1])
plt.axis("off")
plt.title("COCO pretrained YOLO (before wildlife transfer learning)")

**학생 과제 3 — P3 Lab 개념 연결**

- 이미지 *분류*와 객체 *탐지*에서 예측 결과가 어떻게 다른가?
- COCO 사전훈련 모델을 네 클래스 데이터에 그대로 적용하는 경우 무엇을 조심해야 하는가?
- `conf=0.25`를 `0.60`으로 바꿔 탐지 상자 수를 관찰하고 설명하라.

---
## 4. YOLO 전이학습: 동결 전략 비교

Xception에서 사전훈련된 합성곱 기저를 동결하고 새로운 출력층을 학습했던 개념을 YOLO에 적용한다. **두 실험 모두 같은 `yolo11n.pt` 가중치에서 시작**한다.

| 실험 | 설정 | 의미 |
|---|---:|---|
| A (`frozen_10`) | `freeze=10` | 앞쪽 10개 층 동결 |
| B (`full_0`) | `freeze=0` | 모든 층을 학습 대상으로 사용 |

`freeze=10`은 모델의 **앞쪽 열 개 층**을 지정하는 옵션이다. 이것을 곧바로 ‘Backbone 전체 동결’이라고 단정하지 말고 모델 층 구조와 비교하라. 빠른 실행 모드의 짧은 학습 결과는 본 실험 결과로 간주하지 않는다.

**에포크 설정 원칙:** 본 실험의 **25 epoch는 시작 설정이지, 충분히 학습되었다는 보장은 아니다.** 훈련이 끝난 뒤 **검증 mAP50-95 학습곡선**을 보고 필요하면 학습을 더 진행하는 실험을 수행한다. 이때 A/B 모델의 학습 조건을 동일하게 적용해야 동결 전략을 공정하게 비교할 수 있다.

In [ ]:
# 모델별로 같은 설정에서 새로 학습한다. 이 셀은 시간이 걸린다.
EXPERIMENTS = {"frozen_10": 10, "full_0": 0}
checkpoints = {}

for name, freeze_layers in EXPERIMENTS.items():
    model = YOLO(MODEL_NAME)   # 매번 사전훈련 가중치에서 시작
    model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        fraction=FRACTION,
        imgsz=IMGSZ,
        batch=BATCH,
        freeze=freeze_layers,
        lr0=0.001,
        optimizer="AdamW",
        seed=SEED,
        device=DEVICE,
        workers=0,          # Windows/Notebook에서 호환성 개선
        project=str(RUNS),
        name=name,
        exist_ok=True,
        plots=False,
    )
    checkpoints[name] = str(Path(model.trainer.save_dir) / "weights" / "best.pt")

print("Saved models:", checkpoints)

### 학습 결과 확인 — 검증 mAP50-95가 안정화되었는가?

아래 그래프는 각 실험의 **검증 mAP50-95가 epoch에 따라 어떻게 달라지는지** 보여준다. **중요한 것은 에포크 수 자체보다 검증 mAP50-95가 안정화되었는지**이다.

- **마지막까지 상승하는 추세**이면 현재 학습이 충분하지 않을 수 있으므로 epoch를 늘린 실험이 필요하다.
- **여러 epoch 동안 상승폭이 작거나 정체**되어 있다면 더 길게 학습해도 개선이 크지 않을 수 있다.
- 한두 번의 상승·하락만 보고 판단하지 말고 **전체 추세**를 살핀다. 검증 점수는 다소 변동할 수 있다.
- `freeze=10`과 `freeze=0`의 곡선을 비교하여 **학습 속도, 최고 검증 성능, 정체 시점**을 함께 기록한다.

**주의:** 에포크를 늘리는 것만으로 성능 향상이 보장되는 것은 아니다. 과적합 가능성과 계산 비용도 함께 고려한다. 최종 모델 선택과 에포크 조정은 **검증셋**을 근거로 하며, **테스트셋은 튜닝에 이용하지 않는다.**

In [ ]:
plt.figure(figsize=(8, 4))
for name in EXPERIMENTS:
    log = pd.read_csv(RUNS / name / "results.csv")
    log.columns = log.columns.str.strip()
    score_col = [c for c in log.columns if "mAP50-95" in c][0]
    plt.plot(range(1, len(log) + 1), log[score_col], marker="o", label=name)
plt.xlabel("Epoch")
plt.ylabel("Validation mAP50-95")
plt.title("Training comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

#### 본 실험의 epoch를 늘리는 방법

1. 먼저 `QUICK_MODE=False`로 바꾸어 **25 epoch**의 A/B 실험을 수행한다.
2. 바로 위 **검증 mAP50-95 학습곡선**에서 마지막 구간까지 성능이 계속 향상되는지 확인한다.
3. **계속 상승한다면** 기존 실험의 그래프와 `results.csv` 등 결과를 **먼저 별도로 저장**한다. 그다음 환경 설정의 `EPOCHS = 2 if QUICK_MODE else 25`에서 본 실험 값 `25`를 **50 또는 75 등으로 변경**해 A/B를 다시 학습한다. 반대로 충분히 안정화되었다면 무조건 늘릴 필요가 없다.
4. 에포크를 늘린 경우에도 **두 모델 모두 동일한 epoch 상한, 이미지 크기, 학습 데이터 비율, 배치 크기, 초기 가중치 및 시드**를 유지한다. 결과표와 학습곡선을 다시 만들어 비교한다.
5. 보고서에 **최종 설정값, 실제 학습한 에포크 수, 검증 mAP50-95 변화, 추가 학습의 필요성에 대한 판단과 계산 부담**을 기록한다.

> **실행 유의:** 현재 템플릿은 학습 셀을 다시 실행하면 각 모델을 **같은 사전훈련 가중치에서 새로 학습**한다. 즉, `25 → 50`으로 변경해 재실행하는 것은 **25 epoch 체크포인트에서 이어서 학습하는 것(resume)이 아니라 처음부터 최대 50 epoch로 다시 학습**하는 방식이다. 설정을 바꾼 뒤에는 **학습 → 학습곡선 → 검증 비교 → 최종 평가**를 순서대로 다시 실행한다.

**핵심:** *25, 50, 75라는 숫자가 정답인 것이 아니라, 검증 mAP50-95가 충분히 안정화되었는지를 근거로 설명할 수 있어야 한다.*

**학생 과제 4 — 전이학습 실험**

- 학습 전에 어떤 모델이 더 높은 mAP를 보일지 **가설과 이유**를 작성하라.
- 빠른 실행 후 `QUICK_MODE=False`로 바꾸고 A/B를 재학습하라. 훈련 조건(모델, epoch, imgsz, batch, fraction, seed)을 기록하라.
- 두 모델의 실제 epoch 수, **검증 mAP50-95 학습곡선**, 계산 부담을 비교하라. 특히 **마지막 구간까지 mAP50-95가 상승하는지, 안정화되었는지** 판단하고 근거를 제시하라.
- **필수 판단 과제:** 25 epoch에서 학습이 충분하지 않다고 판단되면 **50 또는 75 epoch 등으로 늘려 A/B를 재실험**하라. 충분하다면 추가 학습을 생략한 근거를 설명하라. 에포크를 늘린 경우 **동일한 조건으로 두 모델을 비교**하고 추가 학습 전후 결과를 보고하라.
- **선택 심화:** `freeze=5` 실험을 추가하고 검증셋으로 같은 기준에서 비교하라.

---
## 5. 검증셋 평가: mAP와 개체 수 오차

모델의 `val()`로 **객체 탐지 성능(mAP)**을 평가하고, `predict()`로 검증 사진의 동물 수를 세어 **Count MAE**를 별도로 계산한다.

- **mAP50-95:** 클래스와 위치 예측의 종합적인 탐지 품질, **클수록 좋음**
- **Count MAE:** 사진마다 동물의 종류별 정답 수와 예측 수 차이의 절댓값 평균, **작을수록 좋음**
- 공정한 비교를 위해 **같은 검증셋과 같은 `THRESHOLD=0.35`**를 사용한다.

In [ ]:
def observe(model, folder, threshold=0.35, with_truth=True):
    """폴더의 사진들을 탐지해 사진별 결과와 객체별 결과를 표로 반환한다."""
    image_rows = []
    object_rows = []
    results = model.predict(source=str(folder), conf=threshold, imgsz=IMGSZ,
                            device=DEVICE, stream=True, verbose=False)

    for result in results:
        path = Path(result.path)
        pred_ids = [int(x) for x in result.boxes.cls.cpu().tolist()]
        true_ids = read_ids(path) if with_truth else []

        row = {"image": path.name, "pred_total": len(pred_ids)}
        if with_truth:
            row["true_total"] = len(true_ids)
        for i, name in enumerate(CLASS_NAMES):
            row["pred_" + name] = pred_ids.count(i)
            if with_truth:
                row["true_" + name] = true_ids.count(i)
        image_rows.append(row)

        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().tolist()
            object_rows.append({
                "image": path.name,
                "class": CLASS_NAMES[int(box.cls.item())],
                "confidence": float(box.conf.item()),
                "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            })

    object_columns = ["image", "class", "confidence", "x1", "y1", "x2", "y2"]
    return pd.DataFrame(image_rows), pd.DataFrame(object_rows, columns=object_columns)


def count_mae(df):
    """네 동물 종류에 대한 사진별 개체 수 MAE의 평균."""
    errors = []
    for name in CLASS_NAMES:
        error = (df["true_" + name] - df["pred_" + name]).abs().mean()
        errors.append(error)
    return sum(errors) / len(errors)

In [ ]:
# A/B 모델의 검증 mAP와 Count MAE를 같은 표로 비교
model_rows = []
for name, checkpoint in checkpoints.items():
    model = YOLO(checkpoint)
    metric = model.val(data=DATA_YAML, split="val", imgsz=IMGSZ,
                       batch=BATCH, device=DEVICE, verbose=False, plots=False)
    val_images, _ = observe(model, Path(info["val"]), threshold=THRESHOLD)
    model_rows.append({
        "model": name,
        "freeze": EXPERIMENTS[name],
        "mAP50": float(metric.box.map50),
        "mAP50-95": float(metric.box.map),
        "Count MAE": count_mae(val_images),
    })

comparison = pd.DataFrame(model_rows).sort_values("mAP50-95", ascending=False)
comparison.to_csv(OUT / "validation_comparison.csv", index=False)
display(comparison)
comparison.plot(x="model", y=["mAP50-95", "Count MAE"], kind="bar", rot=0,
                title="Validation metrics (different scales)", figsize=(8, 4))
plt.tight_layout()

**학생 과제 5 — 프로젝트의 핵심 연구 질문**

- 동일 검증셋에서 **mAP50-95 순위와 Count MAE 순위**가 일치하는가? 실제 수치를 제시하라.
- 왜 mAP가 더 높아도 특정 동물의 수를 잘못 셀 수 있는가? 구체적인 예를 설명하라.
- 위 그래프는 **서로 다른 척도를 같은 축에 표시**한다. 이를 개선하는 시각화를 직접 제안하고 만들어보라.
- 필요하다면 `THRESHOLD`를 바꿔 두 모델을 다시 평가해보되, **테스트셋을 설정 선택에 사용하지 말 것.**

---
## 6. 검증셋에서 최종 모델과 신뢰도 임곗값 선택

먼저 **검증 mAP50-95가 높은 모델**을 선택한다. 그 모델에 대해 신뢰도 임곗값을 바꾸면서 Count MAE를 비교한다. 선택 기준을 보고서에 명확히 적고, **테스트셋에는 선정된 설정을 그대로 적용**한다.

In [ ]:
best_name = comparison.iloc[0]["model"]
best_model = YOLO(checkpoints[best_name])

threshold_rows = []
for confidence in [0.25, 0.35, 0.50]:
    val_images, _ = observe(best_model, Path(info["val"]), threshold=confidence)
    threshold_rows.append({"threshold": confidence, "Count MAE": count_mae(val_images)})

threshold_table = pd.DataFrame(threshold_rows).sort_values(["Count MAE", "threshold"])
best_threshold = float(threshold_table.iloc[0]["threshold"])
print("Selected model:", best_name, "| Confidence threshold:", best_threshold)
display(threshold_table)

# 선택 기록: 테스트 결과를 보고 수정하면 안 된다.
selection = {"model": best_name, "threshold": best_threshold,
             "checkpoint": checkpoints[best_name]}
(OUT / "selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")

**학생 과제 6 — 임곗값 선택의 의미**

- 신뢰도 임곗값이 높아지면 **오탐과 미탐**이 각각 어떻게 변할 수 있는가?
- 세 임곗값의 Count MAE가 같은 경우, 어떤 규칙으로 최종 값을 고르는 것이 재현성 측면에서 타당한가?
- **심화:** mAP가 아닌 Count MAE를 최우선 기준으로 모델을 선택하면 결론이 달라지는지 검토하라. 단, 최종 테스트 전 검증 단계에서만 결정해야 한다.

---
## 7. 최종 테스트와 관측 시스템

이제 **선택한 모델과 신뢰도 임곗값을 바꾸지 않고** 독립 테스트셋에서 성능을 평가한다. 사진별·객체별 분석 CSV를 생성한다. (새로운 실험을 위해 모델 설정을 변경했다면 테스트 결과가 아니라 검증 단계부터 다시 진행한다.)

In [ ]:
final_model = YOLO(selection["checkpoint"])
test_metric = final_model.val(data=DATA_YAML, split="test", imgsz=IMGSZ,
                              batch=BATCH, device=DEVICE, verbose=False, plots=False)
test_images, test_objects = observe(final_model, Path(info["test"]),
                                    threshold=best_threshold)

# 실제 관측 결과를 재사용할 수 있게 저장
test_images.to_csv(OUT / "test_image_observations.csv", index=False)
test_objects.to_csv(OUT / "test_object_detections.csv", index=False)
print("Test mAP50-95:", round(float(test_metric.box.map), 4))
print("Test Count MAE:", round(count_mae(test_images), 4))
print("CSV folder:", OUT)
display(test_images.head())

In [ ]:
# 동물 종류별 객체 수 / 출현 사진 수 / 출현 비율
rows = []
for name in CLASS_NAMES:
    n_images = int((test_images["pred_" + name] > 0).sum())
    rows.append({
        "Class": name,
        "Predicted objects": int(test_images["pred_" + name].sum()),
        "Images with class": n_images,
        "Image rate": n_images / len(test_images),
        "Count MAE": (test_images["true_" + name] - test_images["pred_" + name]).abs().mean(),
    })
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "monitoring_summary.csv", index=False)
display(summary)
summary.plot.bar(x="Class", y=["Predicted objects", "Images with class"],
                 figsize=(9, 4), rot=0, title="Wildlife monitoring summary")
plt.tight_layout()
plt.savefig(OUT / "monitoring_summary.png", dpi=150)

**학생 과제 7 — 응용 시스템 확대**

- **서로 다른 통계 세 가지**(종류별 탐지 객체 수 / 해당 종류가 있는 사진 수 / 출현 사진 비율)를 정확히 구분하여 해석하라.
- 두 CSV 중 **이미지 1행**인 파일과 **탐지 박스 1행**인 파일은 무엇인가?
- **필수 확장:** 동물 두 종류가 **같은 사진에서 함께 탐지된 횟수**를 `test_images`에서 계산하고, 자신의 그래프를 하나 추가하라. AI를 활용해도 좋다.
- **선택 확장:** 같은 이미지를 임곗값을 다르게 분석해 출현 통계가 어떻게 달라지는지 보고하라.

---
## 8. 오류 사례 분석

종류별 개체 수 차이가 큰 사진을 찾아 원본·예측 박스를 확인한다. 수량이 같아도 잘못된 박스를 탐지할 수 있으므로 **숫자와 시각적인 증거를 함께** 사용한다.

In [ ]:
# 사진마다 네 종류의 개체 수 오류를 합산
error = 0
for name in CLASS_NAMES:
    error = error + (test_images["true_" + name] - test_images["pred_" + name]).abs()
errors = test_images.copy()
errors["count_error"] = error
worst = errors.sort_values("count_error", ascending=False).head(3)
display(worst[["image", "true_total", "pred_total", "count_error"]])

# 가장 오류가 큰 사진 3장 시각화
fig, axes = plt.subplots(3, 2, figsize=(11, 12))
for i, (_, row) in enumerate(worst.iterrows()):
    path = next(p for p in images["test"] if p.name == row["image"])
    pred = final_model.predict(str(path), conf=best_threshold, imgsz=IMGSZ,
                               device=DEVICE, verbose=False)[0]
    axes[i, 0].imshow(Image.open(path))
    axes[i, 0].set_title("Original | GT: " + str(row["true_total"]))
    axes[i, 1].imshow(pred.plot()[:, :, ::-1])
    axes[i, 1].set_title("Prediction | Pred: " + str(row["pred_total"]))
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "error_examples.png", dpi=150)

**학생 과제 8 — 오류를 설명하기**

오류 사진을 **3장 이상** 골라 이미지 파일명, 정답 개체 수, 예측 개체 수, 관찰 가능한 오류 유형을 표로 기록하라. 가림, 작은 객체, 클래스 혼동, 배경 오탐, 중복 탐지 등을 검토한다. **오류의 원인에 대한 추측과 관찰된 사실을 구분**하라.

---
## 9. 새 관측 사진 적용하기

`P3_wildlife_project/incoming_images/` 폴더에 `.jpg` 또는 `.png` 사진을 넣으면 아래 셀이 자동으로 탐지 CSV를 생성한다. **기본 상태(빈 폴더)에서도 오류 없이 실행된다.** 새 사진에 정답이 없으면 탐지 결과의 실제 정확도를 단정할 수 없다.

In [ ]:
incoming = PROJECT / "incoming_images"
incoming.mkdir(parents=True, exist_ok=True)
new_photos = [p for p in incoming.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]]

if new_photos:
    new_images, new_objects = observe(final_model, incoming, threshold=best_threshold,
                                      with_truth=False)
    new_images.to_csv(OUT / "new_image_observations.csv", index=False)
    new_objects.to_csv(OUT / "new_object_detections.csv", index=False)
    display(new_images)
else:
    print("새 사진을 분석하려면 여기에 이미지 파일을 넣으세요:", incoming)

**학생 과제 9 — 직접 만든 응용 시스템**

- **필수:** 본인이 준비한 새 사진을 3장 이상 분석하고, 실제 이미지와 탐지 결과를 비교하라. 출처와 사용 허용 여부를 확인하라.
- **선택:** 코끼리가 감지된 사진 목록을 별도 CSV로 저장하는 **출현 알림** 기능을 만들어보라. `new_images`의 `pred_elephant` 열을 활용할 수 있다.
- **심화:** 카메라 트랩의 실제 촬영 시간·위치 정보가 주어지면 어떤 분석이 가능해지는지, 현재 데이터만으로는 무엇을 할 수 없는지 구분하라.

---
## 10. 동영상 객체 추적을 이용한 관측 시스템 확장 (선택 심화)

**목표:** 1~9장에서 학습하고 검증한 YOLO 모델을 새로 훈련하지 않고 그대로 사용하여, 연속된 영상에서 동물의 종류·위치뿐 아니라 **추적 ID(track ID)**를 기록한다. Ultralytics의 `model.track()`과 **ByteTrack** 추적기를 사용한다.

**이미지 탐지 vs 동영상 추적**

- **이미지 탐지:** 한 프레임에서 감지된 동물 박스 수를 센다. 같은 동물이 100프레임에 나타나면 박스가 100번 기록될 수 있다.
- **동영상 추적:** 연속 프레임에서 같은 동물에 같은 ID를 부여하려고 한다. 따라서 **프레임별 박스 관측 횟수의 합**과 **영상에서 관측된 고유 추적 ID 수**를 구분할 수 있다.
- **주의:** 추적 도중 ID가 바뀌거나 서로 다른 동물이 같은 ID로 잘못 연결되면 고유 ID 수가 실제 동물 수와 달라진다. 또한 **영상 속 고유 ID 수 ≠ 서식지 전체 개체군 수**이다.

**영상 준비:** 아래 셀에서 **얼룩말 예제 영상을 자동 다운로드**한다. 자신이 준비한 영상이 있으면 `P3_wildlife_project/incoming_videos/` 폴더에 `.mp4`, `.webm`, `.mov`, `.avi` 또는 `.mkv` 파일을 넣는다. **사용자 영상이 있으면 예제보다 우선 선택한다.** 원하는 영상을 직접 지정하는 방법도 아래에 있다. African Wildlife 데이터셋에는 동영상이 없으므로 예제는 별도 자료를 사용한다.

**실행 방법:** 위 본 실험(`QUICK_MODE=False`)을 완료한 뒤 이 단원을 실행한다. 예제 영상 다운로드에 실패하고 사용자 영상도 없으면 **오류 없이 건너뛴다.** `QUICK_MODE=True` 결과는 작동 확인용으로만 사용한다.

참고: [Ultralytics 객체 추적 설명서](https://docs.ultralytics.com/ko/modes/track/) · [추적 결과 `Boxes.id`](https://docs.ultralytics.com/reference/engine/results/)


### 10.0. 야생동물 추적 예제 영상 자동 다운로드

처음 실습할 때는 **얼룩말 두 마리가 나오는 약 19초 영상**을 자동으로 다운로드한다. 영상이 이미 저장되어 있으면 다시 받지 않는다. 자신의 영상을 폴더에 넣으면 **그 영상을 먼저 사용**한다. 동영상이 준비되지 않아도 1~9장의 정지 이미지 프로젝트는 그대로 실행할 수 있다.

- **영상:** [Zebras in a zoo (Equus quagga)](https://commons.wikimedia.org/wiki/File:Zebras_in_a_zoo_%28Equus_quagga%29.webm), Wikimedia Commons
- **제작자:** The Nature Box · **라이선스:** [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/) (출처·저작자 표시, 변경·공유 시 라이선스 준수)
- **내용:** 동물원의 얼룩말 영상이다. 야생 서식지에서 촬영한 영상은 아니므로 **현장 야생동물 관측 성능을 검증하는 데이터**로 해석하지 않는다.
- **주의:** 예제 동영상에는 추적 ID의 정답이 없다. `Unique track IDs`가 실제 동물 수와 일치하는지는 직접 영상을 확인해야 한다.

다운로드를 원하지 않으면 `DOWNLOAD_EXAMPLE_VIDEO = False`로 바꿀 수 있다. 다운로드 실패 시 인터넷 연결을 확인하거나 직접 `.mp4` 영상을 `incoming_videos` 폴더에 넣으면 된다.

In [ ]:
# 공개된 얼룩말 예제 영상 다운로드 (동영상이 없어도 전체 노트북은 실행 가능)
from urllib.request import Request, urlopen
import shutil

VIDEO_DIR = PROJECT / "incoming_videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

DOWNLOAD_EXAMPLE_VIDEO = True
EXAMPLE_VIDEO = VIDEO_DIR / "example_zebras.webm"
EXAMPLE_URL = (
    "https://commons.wikimedia.org/wiki/Special:Redirect/file/"
    "Zebras_in_a_zoo_%28Equus_quagga%29.webm"
)
VIDEO_EXTENSIONS = {".mp4", ".webm", ".mov", ".avi", ".mkv"}

# 학생이 준비한 동영상이 있다면 예제 다운로드를 생략한다.
custom_videos = sorted(p for p in VIDEO_DIR.iterdir()
                       if p.suffix.lower() in VIDEO_EXTENSIONS and p != EXAMPLE_VIDEO)

if custom_videos:
    print("사용자 동영상이 있어 예제 다운로드를 생략합니다:", custom_videos[0].name)
elif EXAMPLE_VIDEO.exists() and EXAMPLE_VIDEO.stat().st_size > 100_000:
    print("이미 다운로드된 예제 동영상 사용:", EXAMPLE_VIDEO)
elif DOWNLOAD_EXAMPLE_VIDEO:
    try:
        request = Request(EXAMPLE_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(request, timeout=60) as response, EXAMPLE_VIDEO.open("wb") as output:
            shutil.copyfileobj(response, output)
        if EXAMPLE_VIDEO.stat().st_size < 100_000:
            raise ValueError("다운로드된 영상의 크기가 너무 작습니다.")
        print("예제 영상 다운로드 완료:", EXAMPLE_VIDEO)
    except Exception as exc:
        EXAMPLE_VIDEO.unlink(missing_ok=True)
        print("예제 영상 다운로드 실패:", exc)
        print("인터넷 연결을 확인하거나 incoming_videos에 MP4 영상을 넣으세요.")
else:
    print("예제 다운로드를 사용하지 않습니다. 직접 영상 파일을 넣어주세요.")

In [ ]:
# 동영상 입력: 직접 준비한 영상이 있으면 우선 사용하고, 없으면 다운로드한 예제를 사용한다.
import cv2

custom_videos = sorted(p for p in VIDEO_DIR.iterdir()
                       if p.suffix.lower() in VIDEO_EXTENSIONS and p != EXAMPLE_VIDEO)
VIDEO_PATH = custom_videos[0] if custom_videos else (EXAMPLE_VIDEO if EXAMPLE_VIDEO.exists() else None)
# 다른 파일을 사용하려면 직접 지정할 수 있다. 예: VIDEO_PATH = VIDEO_DIR / "my_video.mp4"
TRACKER = "bytetrack.yaml"            # 선택 심화: "botsort.yaml"과 비교
TRACK_CONF = best_threshold           # 검증셋에서 선택한 탐지 신뢰도 사용

RUN_VIDEO_TRACKING = False
VIDEO_FPS = 0
VIDEO_WIDTH, VIDEO_HEIGHT = 0, 0

if VIDEO_PATH is None:
    print("분석할 동영상이 없습니다. 아래 폴더에 짧은 영상을 넣어주세요:", VIDEO_DIR)
else:
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    if cap.isOpened():
        VIDEO_FPS = cap.get(cv2.CAP_PROP_FPS) or 0
        VIDEO_WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        VIDEO_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        RUN_VIDEO_TRACKING = True
        print("선택한 영상:", VIDEO_PATH.name)
        print("크기:", VIDEO_WIDTH, "x", VIDEO_HEIGHT,
              "| FPS:", round(VIDEO_FPS, 2), "| 추적기:", TRACKER)
    else:
        print("영상 파일을 열 수 없습니다. 파일 형식/코덱을 확인하세요:", VIDEO_PATH)
    cap.release()


### 10.1. 학습된 모델로 객체 추적하기

이 코드는 `model.track()`의 결과에서 `boxes.cls`, `boxes.conf`, `boxes.xywh`, `boxes.id`를 읽어 **객체별 추적 CSV**와 **프레임별 요약 CSV**를 저장한다. 추적 결과에 ID가 없을 수 있으므로 그 경우 빈 값으로 기록한다.

**중요:** 한 파일 전체를 `source=`로 넘겨 처리하므로 이 실습에서는 `persist=True`를 직접 지정할 필요가 없다. 나중에 OpenCV로 프레임을 직접 하나씩 전달하는 방식으로 바꾼다면 **같은 영상의 연속 프레임에 한해서** `persist=True`를 사용한다.

영상이 길면 시간이 오래 걸릴 수 있다. 초기 실습에는 짧은 영상 하나를 사용한다.

In [ ]:
# 객체별/프레임별 기록. 영상을 넣지 않은 경우 자동으로 건너뛴다.
track_columns = ["frame", "seconds", "class", "track_id", "confidence", "x", "y"]
frame_columns = ["frame", "seconds", "detections", "tracked"]
tracking_objects = pd.DataFrame(columns=track_columns)
tracking_frames = pd.DataFrame(columns=frame_columns)

if RUN_VIDEO_TRACKING:
    tracking_model = YOLO(selection["checkpoint"])  # P3에서 선택한 모델 재사용
    video_results = tracking_model.track(
        source=str(VIDEO_PATH), tracker=TRACKER, conf=TRACK_CONF,
        imgsz=IMGSZ, device=DEVICE, stream=True, save=True,
        project=str(OUT), name="tracking_video", exist_ok=True, verbose=False,
    )
    object_rows, frame_rows = [], []
    for frame_no, result in enumerate(video_results, start=1):
        boxes = result.boxes
        n_boxes = len(boxes) if boxes is not None else 0
        ids = (boxes.id.int().cpu().tolist()
               if boxes is not None and boxes.is_track else [None] * n_boxes)
        seconds = (frame_no - 1) / VIDEO_FPS if VIDEO_FPS > 0 else None
        frame_rows.append({"frame": frame_no, "seconds": seconds,
                           "detections": n_boxes, "tracked": sum(i is not None for i in ids)})
        if boxes is not None:
            for box, track_id in zip(boxes, ids):
                x, y, _, _ = box.xywh[0].cpu().tolist()
                object_rows.append({"frame": frame_no, "seconds": seconds,
                                    "class": CLASS_NAMES[int(box.cls.item())],
                                    "track_id": track_id,
                                    "confidence": float(box.conf.item()),
                                    "x": x, "y": y})
    tracking_objects = pd.DataFrame(object_rows, columns=track_columns)
    tracking_frames = pd.DataFrame(frame_rows, columns=frame_columns)
    tracking_objects.to_csv(OUT / "video_object_tracks.csv", index=False)
    tracking_frames.to_csv(OUT / "video_frame_summary.csv", index=False)
    print("분석한 프레임:", len(tracking_frames), "| 객체 관측 행:", len(tracking_objects))
    print("CSV:", OUT / "video_object_tracks.csv")
    print("추적 결과 영상 저장 폴더:", OUT / "tracking_video")
else:
    print("동영상 추적 건너뜀: 기존 정지 이미지 P3 결과에는 영향이 없습니다.")


### 10.2. 종류별 탐지 관측 횟수와 추적 ID 수 비교

**프레임별 박스 관측 횟수**는 여러 프레임에서 같은 개체가 반복 등장하면 계속 증가한다. **고유 추적 ID 수**는 같은 ID의 반복 등장을 한 번만 센다. 아래 표는 네 동물 종류를 모두 포함하며, 해당 종류가 관측되지 않으면 0으로 표시한다.

이 수치는 **추적 알고리즘의 결과**이지, 정답 주석이 있는 동영상으로 검증한 실제 개체 수가 아니다.

In [ ]:
# 프레임별 탐지 누적 횟수와 고유 추적 ID 수 비교
if RUN_VIDEO_TRACKING:
    summary_rows = []
    for name in CLASS_NAMES:
        rows = tracking_objects[tracking_objects["class"] == name]
        summary_rows.append({
            "Class": name,
            "Box observations": len(rows),
            "Frames with class": rows["frame"].nunique(),
            "Unique track IDs": rows["track_id"].nunique(),
        })
    tracking_summary = pd.DataFrame(summary_rows)
    tracking_summary.to_csv(OUT / "video_tracking_summary.csv", index=False)
    display(tracking_summary)
    tracking_summary.plot(x="Class", y=["Box observations", "Unique track IDs"],
                          kind="bar", rot=0, figsize=(8, 4), title="Video observations vs unique tracks")
    plt.tight_layout()
    plt.savefig(OUT / "video_tracking_summary.png", dpi=150)
    plt.show()
else:
    print("영상이 없으므로 추적 집계표를 생성하지 않았습니다.")


### 10.3. 이동 궤적 시각화 (참고 예제)

각 추적 ID의 경계 상자 중심 좌표를 시간 순서대로 이어서 **영상 화면 위의 이동 궤적**을 표시한다. 가장 오래 관측된 ID를 최대 5개까지 그린다. 좌표는 영상의 **픽셀 좌표**이므로, 카메라가 움직인다면 이 선을 동물의 실제 지상 이동 경로로 해석할 수 없다.

In [ ]:
# 관측 프레임 수가 많은 ID 최대 5개의 이동 궤적 그리기
if RUN_VIDEO_TRACKING and tracking_objects["track_id"].notna().any():
    valid_tracks = tracking_objects.dropna(subset=["track_id"])
    top_ids = valid_tracks["track_id"].value_counts().head(5).index
    plt.figure(figsize=(8, 5))
    for track_id in top_ids:
        points = valid_tracks[valid_tracks["track_id"] == track_id]
        plt.plot(points["x"], points["y"], marker=".", markersize=3,
                 label=f"ID {int(track_id)} ({points.iloc[0]['class']})")
    plt.xlim(0, VIDEO_WIDTH)
    plt.ylim(VIDEO_HEIGHT, 0)  # 이미지의 y축은 아래로 증가
    plt.xlabel("x (pixels)")
    plt.ylabel("y (pixels)")
    plt.title("Tracked paths in image coordinates")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT / "video_track_paths.png", dpi=150)
    plt.show()
else:
    print("표시할 추적 ID가 없습니다. 영상의 동물 종류와 탐지 성능을 확인하세요.")


### 학생 과제 10 — 동영상 객체 추적을 이용한 응용 시스템 (선택 심화)

**심화 수행 시 기본 과제 (1~4)**

1. **영상 준비:** 다운로드된 얼룩말 예제 또는 본인이 준비한 영상을 사용한다. 등장 동물 종류, 영상 길이, FPS, 영상 출처·사용 조건을 기록한다. P3의 네 학습 클래스와 영상의 동물 종류가 일치하는지 확인한다.
2. **추적 실행:** 최종 YOLO 가중치를 이용해 `model.track()`을 실행하고 ID가 표시된 결과 영상을 저장한다. `predict()`와 `track()`의 출력에서 무엇이 달라지는지 설명한다.
3. **집계 비교:** `video_tracking_summary.csv`에서 종류별 **프레임별 박스 관측 횟수**와 **고유 ID 수**를 비교하고, 차이가 발생하는 이유를 실제 영상과 연결해 설명한다.
4. **오류 검증:** 영상의 서로 다른 시점 3곳 이상에서 ID가 유지되는지 눈으로 확인한다. **ID 변경(ID switch), 중간에 ID가 사라지는 현상, 잘못된 클래스·탐지 누락** 중 실제로 관찰한 사례를 제시한다. 정답 주석이 없다면 정확한 추적 성능 수치를 계산했다고 주장하지 않는다.

**추가 선택 과제 (5~6)**

5. `video_track_paths.png`에서 주요 개체의 화면 내 이동 경로를 해석하고, 카메라 이동 때문에 나타날 수 있는 오류를 설명한다. 필요하면 다른 시각화를 개발한다.
6. **가상의 구역 통과선**을 하나 지정하고, 특정 방향으로 통과한 개체의 ID를 한 번만 세는 기능을 개발한다. 중복 계수를 방지하는 방법을 설명한다.

**탐구 확장:** `TRACKER="botsort.yaml"`로 다시 실행해 ByteTrack과 결과를 비교한다. 두 방법의 고유 ID 수가 다르다면 **왜 다른지 영상과 근거로 분석**하되, 어느 쪽이 정확한지는 정답 ID 주석이나 별도의 수작업 검증 없이는 단정하지 않는다.

**제출(심화 선택자):** 영상 정보·추적 결과 영상·객체별 CSV·요약 CSV·주요 추적 오류 이미지 또는 프레임·분석 결과를 포함한다. AI가 작성한 추가 코드는 실행 결과를 검증하고 설명할 수 있어야 한다.

---
## 11. 프로젝트 보고서 및 제출 점검

### 보고서 권장 목차

1. **문제 정의:** 객체 탐지 결과로 해결할 실제 관측 업무
2. **강의 연계:** CNN, Faster R-CNN, Xception 전이학습과 YOLO 실험의 관계
3. **데이터:** 네 클래스와 train/val/test 개수 및 라벨 구조
4. **실험 설계:** 같은 사전훈련 가중치에서 `freeze=10` vs `freeze=0`; 모든 공통 조건, GPU/CPU, 실제 epoch, **검증 mAP50-95 학습곡선의 안정화 여부와 epoch 연장 판단**
5. **검증:** 두 모델의 val mAP50-95, val Count MAE, 임곗값 선택 근거 (**본 실험 결과만 사용**)
6. **자동 관측:** 테스트 mAP, Count MAE, 클래스별 객체 수·출현 사진 수·출현 비율과 결과 CSV
7. **오류 분석:** 대표 이미지 3장과 가능 원인(관측 vs 추측 구분)
8. **확장 기능:** 동시 출현 통계(필수), 새 사진 분석(필수), 알림 기능(선택)
9. **결론·한계:** “mAP가 높으면 Count MAE도 낮은가?”에 실제 표로 답하기. 사진의 탐지 수 ≠ 실제 개체군 규모
10. **동영상 객체 추적(선택 심화):** 영상 정보, 프레임별 관측 횟수와 고유 추적 ID 수, 영상에서 확인한 ID 오류, CSV/추적 영상/궤적 그래프
11. **AI 활용 기록:** AI 제안, 실행 결과, 학생 본인의 검증·수정 사례 3개

### 제출 전 확인

- [ ] QUICK_MODE 실행 뒤 **본 실험 (`QUICK_MODE=False`)** 완료
- [ ] 모델 A/B를 같은 초기 가중치·데이터 분할에서 학습하고 검증
- [ ] **검증 mAP50-95가 안정화되었는지 학습곡선으로 확인**하고, 필요하면 epoch를 늘려 동일 조건으로 재실험하거나 연장하지 않은 이유를 기록
- [ ] 같은 검증셋에서 mAP와 Count MAE를 **모두 비교**
- [ ] 최종 모델·임곗값을 검증셋에서 정한 뒤 테스트는 한 번만 사용
- [ ] CSV, 관측 통계, 오류 사례, 동시 출현 분석 및 새 사진 분석을 제출
- [ ] 동영상 심화 수행자는 **동영상 추적 CSV·영상 결과·ID 오류 분석** 제출 (미수행자는 생략)
- [ ] AI 활용 내용과 직접 검증 과정 기록

**추가 코드를 작성할 수 있지만, 기본 템플릿은 한 줄도 채우지 않고 실행할 수 있어야 한다.**

#### 참고
- [African Wildlife 데이터](https://docs.ultralytics.com/datasets/detect/african-wildlife/)
- [YOLO11](https://docs.ultralytics.com/models/yolo11/)
- [학습](https://docs.ultralytics.com/modes/train/) · [검증](https://docs.ultralytics.com/modes/val/) · [추론](https://docs.ultralytics.com/modes/predict/) · [객체 추적](https://docs.ultralytics.com/ko/modes/track/)